# 04 · PCA, UMAP and Leiden Clustering

**Goal**: Perform PCA, construct a gene expression neighborhood graph, and apply UMAP and Leiden clustering to identify expression-based groups of spatial bins.

**Input**: results/adata/03_normalized.h5ad

**Output**: results/adata/04_clustered.h5ad, clustering figures, and summary tables

**Note**: Clustering uses the highly variable genes (HVGs) selected in Step 3. All genes and original raw counts are retained. Clusters represent groups of spatial bins, not individual cell types.

## 1 · Imports, dependency check and project paths

In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import sys
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import matplotlib.pyplot as plt

for package in ("igraph", "leidenalg", "umap-learn"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError as exc:
        raise RuntimeError(
            f"Missing {package}. In a separate terminal run: "
            "conda install -c conda-forge python-igraph leidenalg umap-learn -y "
            "and restart the Jupyter kernel."
        ) from exc

def setup_clustering_paths():
    cwd = Path.cwd().resolve()
    ROOT = next((p for p in (cwd, *cwd.parents)
                 if (p / "data").is_dir() and (p / "results").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Run this notebook from visium_hd_tutorial or its notebooks/ folder")
    INPUT = ROOT / "results/adata/03_normalized.h5ad"
    OUTPUT = ROOT / "results/adata/04_clustered.h5ad"
    FIGURES = ROOT / "results/figures"
    TABLES = ROOT / "results/tables"
    FIGURES.mkdir(parents=True, exist_ok=True)
    TABLES.mkdir(parents=True, exist_ok=True)
    if not INPUT.is_file():
        raise FileNotFoundError(f"Missing previous pipeline output: {INPUT}")

    return ROOT, INPUT, OUTPUT, FIGURES, TABLES

ROOT, INPUT, OUTPUT, FIGURES, TABLES = setup_clustering_paths()
print("Python:", sys.executable)
print("Scanpy:", version("scanpy"))
print("Project root:", ROOT)

## 2 · Load normalized AnnData and validate inputs

Load the normalized AnnData file from Step 3 and verify that the gene expression data, original raw counts, spatial coordinates, and highly variable genes (HVGs) are correctly preserved for clustering.

In [ ]:
def load_and_validate_normalized(input_path):
    adata = sc.read_h5ad(input_path)
    assert adata.n_obs > 2 and adata.n_vars > 2
    assert adata.obs_names.is_unique and adata.var_names.is_unique
    assert sparse.issparse(adata.X), "Expected sparse log-normalized expression"
    assert "counts" in adata.layers and sparse.issparse(adata.layers["counts"])
    assert "highly_variable" in adata.var
    assert "spatial" in adata.obsm and adata.obsm["spatial"].shape == (adata.n_obs, 2)
    assert np.isfinite(adata.obsm["spatial"]).all()
    assert np.isfinite(adata.X.data).all()
    assert np.isfinite(adata.layers["counts"].data).all()
    assert (adata.layers["counts"].data >= 0).all()
    n_hvg = int(adata.var["highly_variable"].sum())
    assert n_hvg >= 3, f"Insufficient HVGs: {n_hvg}"
    print(adata)
    print("HVGs:", n_hvg)
    print("Count layer:", adata.layers["counts"].shape)
    print("Spatial coordinates:", adata.obsm["spatial"].shape)
    return adata, n_hvg

adata, n_hvg = load_and_validate_normalized(INPUT)

## 3 · PCA on HVGs

Perform principal component analysis (PCA) using highly variable genes (HVGs) to reduce the dimensionality of gene expression data and identify major patterns of variation. Visualize the explained variance of each principal component.

In [ ]:
def run_pca(adata, n_hvg, figures, seed=42):
    N_PCS = min(30, n_hvg - 1, adata.n_obs - 1)
    assert N_PCS >= 2
    sc.pp.pca(adata, n_comps=N_PCS, mask_var="highly_variable", svd_solver="arpack", random_state=seed)
    assert adata.obsm["X_pca"].shape == (adata.n_obs, N_PCS)
    assert np.isfinite(adata.obsm["X_pca"]).all()
    variance_ratio = adata.uns["pca"]["variance_ratio"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(np.arange(1, len(variance_ratio) + 1), variance_ratio, marker="o", markersize=3)
    ax.set(xlabel="Principal component", ylabel="Explained variance ratio", title="PCA variance explained")
    fig.tight_layout()
    fig.savefig(figures / "04_pca_variance.png", dpi=180)
    plt.show()
    print("PCA shape:", adata.obsm["X_pca"].shape)
    return N_PCS

SEED = 42
N_PCS = run_pca(adata, n_hvg, FIGURES, seed=SEED)

## 4 · Expression neighbors, UMAP and Leiden clustering

Construct a neighborhood graph based on gene expression similarity, visualize spatial bins using UMAP, and identify groups of bins with similar expression patterns using Leiden clustering.

In [ ]:
def run_expression_clustering(adata, n_pcs, seed=42, n_neighbors=15, leiden_resolution=0.5):
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs, use_rep="X_pca", random_state=seed)
    sc.tl.umap(adata, random_state=seed)
    sc.tl.leiden(adata, resolution=leiden_resolution, key_added="leiden", flavor="igraph", directed=False, random_state=seed)
    assert adata.obsm["X_umap"].shape == (adata.n_obs, 2)
    assert np.isfinite(adata.obsm["X_umap"]).all()
    assert "connectivities" in adata.obsp and "distances" in adata.obsp
    print("Clusters:", adata.obs["leiden"].nunique())
    print(adata.obs["leiden"].value_counts().sort_index().to_string())

N_NEIGHBORS = 15
LEIDEN_RESOLUTION = 0.5
run_expression_clustering(adata, N_PCS, SEED, N_NEIGHBORS, LEIDEN_RESOLUTION)

## 5 · UMAP and spatial coordinate-only cluster plots

Visualize Leiden clusters using UMAP and spatial coordinates to compare gene expression patterns with their distribution across the tissue.

**Note**: Spatial clusters are plotted using coordinates only, without a tissue-image overlay because image registration has not been validated.

In [ ]:
def plot_umap_and_spatial_clusters(adata, figures):
    categories = list(adata.obs["leiden"].cat.categories)
    cluster_idx = adata.obs["leiden"].cat.codes.to_numpy()
    cmap = plt.get_cmap("tab20", max(1, len(categories)))
    colors = cmap(cluster_idx % max(1, len(categories)))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    xy = adata.obsm["X_umap"]
    axes[0].scatter(xy[:, 0], xy[:, 1], c=colors, s=4, rasterized=True)
    axes[0].set(xlabel="UMAP 1", ylabel="UMAP 2", title="Leiden clusters (expression)")
    spatial = adata.obsm["spatial"]
    axes[1].scatter(spatial[:, 0], spatial[:, 1], c=colors, s=5, rasterized=True)
    axes[1].invert_yaxis()
    axes[1].set_aspect("equal", adjustable="datalim")
    axes[1].set(xlabel="Full-resolution X (pixels)", ylabel="Full-resolution Y (pixels)",
                title="Spatial bins (no tissue image overlay)")
    from matplotlib.lines import Line2D
    handles = [Line2D([0], [0], marker="o", linestyle="", markersize=6,
                      markerfacecolor=cmap(i % max(1, len(categories))), markeredgecolor="none",
                      label=str(cat)) for i, cat in enumerate(categories)]
    fig.legend(handles=handles, title="Leiden", loc="center right", bbox_to_anchor=(1.08, 0.5),
               ncol=1 if len(categories) <= 15 else 2, fontsize=8)
    fig.tight_layout()
    fig.savefig(figures / "04_umap_spatial_clusters.png", dpi=180, bbox_inches="tight")
    plt.show()

plot_umap_and_spatial_clusters(adata, FIGURES)

## 6 · Export cluster counts and processing metadata

Calculate the number and proportion of spatial bins in each Leiden cluster, save the cluster summary as a CSV file, and record the clustering parameters for reproducibility.

In [ ]:
def export_cluster_summary(adata, tables, n_hvg, n_pcs, n_neighbors, leiden_resolution, seed):
    summary = (adata.obs["leiden"].value_counts().rename_axis("leiden")
               .reset_index(name="n_bins"))
    summary["fraction"] = summary["n_bins"] / adata.n_obs
    summary.to_csv(tables / "04_cluster_sizes.csv", index=False)
    adata.uns["clustering_pipeline"] = {
        "input": "results/adata/03_normalized.h5ad",
        "hvg_count": int(n_hvg),
        "n_pcs": int(n_pcs),
        "n_neighbors": int(n_neighbors),
        "leiden_resolution": float(leiden_resolution),
        "leiden_flavor": "igraph",
        "random_seed": int(seed),
        "qc_filters_applied": False,
        "image_registration_verified": False,
    }
    print(summary.to_string(index=False))
    return summary

summary = export_cluster_summary(adata, TABLES, n_hvg, N_PCS, N_NEIGHBORS, LEIDEN_RESOLUTION, SEED)

## 7 · Save and verify clustered AnnData

Save the clustering results as an AnnData file, then reload it to verify that PCA, UMAP, Leiden clusters, spatial coordinates, and original raw counts are correctly preserved for downstream analysis.

In [ ]:
def save_and_verify_clusters(adata, output, root):
    output.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(output)
    check = sc.read_h5ad(output)
    assert check.shape == adata.shape
    assert check.obs_names.equals(adata.obs_names)
    assert check.var_names.equals(adata.var_names)
    assert np.array_equal(check.obsm["spatial"], adata.obsm["spatial"])
    assert np.allclose(check.obsm["X_pca"], adata.obsm["X_pca"])
    assert np.allclose(check.obsm["X_umap"], adata.obsm["X_umap"])
    assert check.obs["leiden"].equals(adata.obs["leiden"])
    assert "counts" in check.layers and sparse.issparse(check.layers["counts"])
    assert np.array_equal(check.layers["counts"].data, adata.layers["counts"].data)
    assert "connectivities" in check.obsp and "distances" in check.obsp
    print("PASS — 04_clustering completed")
    print("AnnData:", check.shape)
    print("HVGs:", int(check.var["highly_variable"].sum()))
    print("Leiden clusters:", check.obs["leiden"].nunique())
    print("Output:", output.relative_to(root))
    return check

check = save_and_verify_clusters(adata, OUTPUT, ROOT)

## Next: 05. Spatial Analysis

In `05_spatial_analysis.ipynb`, we will construct a spatial neighborhood graph and analyze the spatial distribution and organization of Leiden clusters.

**Input**: results/adata/04_clustered.h5ad

**Note**: Spatial analysis uses bin coordinates without tissue-image overlays because image registration has not been validated.